# Mealie Integration Setup on Chameleon
**Team: Bias & Variance | proj18 | April 2026**

This notebook provisions a fresh VM on Chameleon Cloud and deploys the full Mealie ML stack from scratch:
1. Creates a new lease on KVM@TACC
2. Launches a new VM with CC-Ubuntu24.04
3. Installs Docker and K3s (Kubernetes)
4. Clones all four team repos
5. Deploys: PostgreSQL, MinIO, MLflow, Inference API, Mealie, CronJobs, Monitoring
6. Bootstraps training data and initial ALS model

> Run cells top to bottom. Each cell is idempotent — safe to re-run.

---
## Cell 1 — Imports and project setup

In [5]:
from chi import server, context, lease, network
import chi, os, time, datetime

context.version = "1.0"
context.choose_project()  # select CHI-251409
context.choose_site(default="KVM@TACC")

username = os.getenv('USER')
print(f"Logged in as: {username}")

Logged in as: sg9469_nyu_edu


## Cell 2 — Create a new lease on KVM@TACC

In [6]:
import datetime

LEASE_NAME = f"proj18-mealie-{username}"
LEASE_DURATION_HOURS = 48  # adjust as needed

l = lease.Lease(
    LEASE_NAME,
    duration=datetime.timedelta(hours=LEASE_DURATION_HOURS)
)
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)
l.show()

print("Lease status:", l.status)
if l.status != "ACTIVE":
    print("WARNING: Lease not ACTIVE yet. Wait a few seconds and re-run this cell.")
else:
    print("Lease is ACTIVE - good to go.")

HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>proj_18_servin…

Lease Details:
Name: proj_18_serving
ID: 6feccb5d-2484-41f4-b1a7-4110762bae41
Status: ACTIVE
Start Date: 2026-04-17 09:59:00
End Date: 2026-05-07 09:58:00
User ID: 55828c46b9d77d1109d40a2300dcf2b735f1ec72a114fbbc86e1e24370664091
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: 833254fc-59df-4262-a484-2393f8016a41, Status: active, Flavor: 833254fc-59df-4262-a484-2393f8016a41, Amount: 1

Events:

Lease status: ACTIVE
Lease is ACTIVE — good to go.


## Cell 3 — Launch VM using the lease

In [8]:
import time
import openstack
from chi import server

conn = openstack.connect(cloud="envvars")

reserved_flavor = l.get_reserved_flavors()[0].name
vm_name = f"node-integration-{username}"

print(f"Using reserved flavor: {reserved_flavor}")
print(f"Creating VM: {vm_name}")

s = server.Server(
    vm_name,
    image_name="CC-Ubuntu24.04",
    flavor_name=reserved_flavor,
)

print("Submitting VM...")
s.submit(idempotent=True)
s.refresh()
server_id = s.id
print(f"Server ID: {server_id}")
print("Polling for ACTIVE status...")

srv = None
for i in range(40):
    try:
        srv = conn.compute.get_server(server_id)
        print(f"[{i}] Status: {srv.status}")
        if srv.status == "ACTIVE":
            print("VM is ACTIVE")
            break
        if srv.status == "ERROR":
            print("VM ERROR:", getattr(srv, 'fault', None))
            raise RuntimeError("VM creation failed.")
    except Exception as e:
        print(f"[{i}] {e}")
    time.sleep(15)

if srv is None or srv.status != "ACTIVE":
    raise RuntimeError(f"VM not ACTIVE after polling. Last status: {getattr(srv, 'status', 'unknown')}")

s = server.Server.from_existing(vm_name)
s.refresh()
s.show(type="widget")

Using reserved flavor: reservation:833254fc-59df-4262-a484-2393f8016a41
Creating VM: node-integration-proj18
Submitting VM...


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-integration-proj18's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-integration-proj18
Id,dafd56b6-387f-45ca-bc07-8a58a50774ac
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:833254fc-59df-4262-a484-2393f8016a41
Addresses,sharednet1: IP: 10.56.1.141 (v4) Type: fixed MAC: fa:16:3e:8b:20:fa IP: 129.114.26.214 (v4) Type: floating MAC: fa:16:3e:8b:20:fa
Network Name,sharednet1
Created At,2026-04-20T07:12:06Z
Keypair,sg9469_nyu_edu-jupyter
Reservation Id,None
Host Id,951daf2a5ba2285d35cce6efd97dfeafbaf2e84b1605a1aaba7bfdd3


Submit returned. Polling status...
[0] Server not visible yet
[1] Server not visible yet
[2] Server not visible yet


KeyboardInterrupt: 

In [ ]:
# Add security groups for SSH and all Kubernetes NodePorts
security_groups = [
    {'name': 'allow-ssh',   'port': 22,    'description': 'SSH access'},
    {'name': 'allow-30090', 'port': 30090, 'description': 'Mealie UI'},
    {'name': 'allow-30800', 'port': 30800, 'description': 'Inference API'},
    {'name': 'allow-30500', 'port': 30500, 'description': 'MLflow'},
    {'name': 'allow-30900', 'port': 30900, 'description': 'MinIO API'},
    {'name': 'allow-30901', 'port': 30901, 'description': 'MinIO Console'},
    {'name': 'allow-30091', 'port': 30091, 'description': 'Prometheus'},
    {'name': 'allow-30300', 'port': 30300, 'description': 'Grafana'},
    {'name': 'allow-30903', 'port': 30903, 'description': 'Alertmanager'},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        'name': sg['name'],
        'description': sg['description'],
    })
    secgroup.add_rule(direction='ingress', protocol='tcp', port=sg['port'])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg['name'])
    print(f"Added security group: {sg['name']}")

print("All security groups attached.")

## Cell 4 — Associate floating IP and verify connectivity

In [4]:
s.associate_floating_ip()
s.refresh()
s.check_connectivity()
s.refresh()

# Extract floating IP for use throughout the notebook
floating_ip = None
for addr_list in s.addresses.values():
    for addr in addr_list:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]
            break

if floating_ip is None:
    raise RuntimeError("Could not detect floating IP. Check s.show() output.")

print("Floating IP:", floating_ip)
print("SSH command: ssh cc@" + floating_ip)
s.show(type="widget")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Checking connectivity to 129.114.26.214 port 22.


Connection successful


Attribute,node-integration-proj18
Id,dafd56b6-387f-45ca-bc07-8a58a50774ac
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:833254fc-59df-4262-a484-2393f8016a41
Addresses,sharednet1: IP: 10.56.1.141 (v4) Type: fixed MAC: fa:16:3e:8b:20:fa IP: 129.114.26.214 (v4) Type: floating MAC: fa:16:3e:8b:20:fa
Network Name,sharednet1
Created At,2026-04-20T07:12:06Z
Keypair,sg9469_nyu_edu-jupyter
Reservation Id,None
Host Id,951daf2a5ba2285d35cce6efd97dfeafbaf2e84b1605a1aaba7bfdd3


## Cell 5 — Verify SSH connectivity

In [5]:
s.check_connectivity()
print(f"VM is reachable at {floating_ip}")

Checking connectivity to 129.114.26.214 port 22.


Connection successful
VM is reachable via SSH


---
## Cell 6 — Install Docker on the VM

Skip this if Docker is already installed (ask Mahima if she already set this up).

In [6]:
# Check if Docker is already installed
result = s.execute("docker --version 2>/dev/null || echo 'NOT_INSTALLED'")
print(result)

if 'NOT_INSTALLED' in str(result):
    print("Installing Docker...")
    s.execute("curl -sSL https://get.docker.com/ | sudo sh")
    s.execute("sudo groupadd -f docker && sudo usermod -aG docker $USER")
    s.execute("sudo systemctl restart docker")
    print("Docker installed.")
else:
    print("Docker already installed — skipping.")

/opt/conda/lib/python3.12/site-packages/paramiko/client.py:885: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.214: b'4da268233d1eb184b2d0bf6e2e24566f'
  warnings.warn(


NOT_INSTALLED
Command exited with status 0.
=== stdout ===
NOT_INSTALLED

(no stderr)
Installing Docker...
# Executing docker install script, commit: 8fb5881103ac6f2fb404605d6d5b1f84244f3896


+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install ca-certificates curl >/dev/null

Running kernel seems to be up-to-date.

Restarting services...
 systemctl restart packagekit.service

No containers need to be restarted.

No user sessions are running outdated binaries.

No VM guests are running outdated hypervisor (qemu) binaries on this host.
+ sh -c install -m 0755 -d /etc/apt/keyrings
+ sh -c curl -fsSL "https://download.docker.com/linux/ubuntu/gpg" -o /etc/apt/keyrings/docker.asc
+ sh -c chmod a+r /etc/apt/keyrings/docker.asc
+ sh -c echo "deb [arch=amd64 signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu noble stable" > /etc/apt/sources.list.d/docker.list
+ sh -c apt-get -qq update >/dev/null
+ sh -c DEBIAN_FRONTEND=noninteractive apt-get -y -qq install docker-ce docker-ce-cli containerd.io docker-compose-plugin docker-ce-rootless-extras docker-buildx-plugin docker-model-plugin >/dev/null

Running kern

  UNIT                                                                           LOAD   ACTIVE SUB       DESCRIPTION
  proc-sys-fs-binfmt_misc.automount                                              loaded active running   Arbitrary Executable File Formats File System Automount Point
  sys-devices-pci0000:00-0000:00:03.0-virtio1-net-ens3.device                    loaded active plugged   Virtio network device
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda1.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda1
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda2.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda2
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda-vda3.device              loaded active plugged   /sys/devices/pci0000:00/0000:00:04.0/virtio2/block/vda/vda3
  sys-devices-pci0000:00-0000:00:04.0-virtio2-block-vda.device                   loaded active

INFO: Docker daemon enabled and started

+ sh -c docker version


tive plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.24/tty/ttyS24
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.25-tty-ttyS25.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.25/tty/ttyS25
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.26-tty-ttyS26.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.26/tty/ttyS26
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.27-tty-ttyS27.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.27/tty/ttyS27
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.28-tty-ttyS28.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.28/tty/ttyS28
  sys-devices-platform-serial8250-serial8250:0-serial8250:0.29-tty-ttyS29.device loaded active plugged   /sys/devices/platform/serial8250/serial8250:0/serial8250:0.29/tty/ttyS29
  sys-devices-platform

## Cell 7 — Install K3s (lightweight Kubernetes)

In [7]:
# Check if K3s is already installed
result = s.execute("kubectl version --client 2>/dev/null || echo 'NOT_INSTALLED'")
print(result)

if 'NOT_INSTALLED' in str(result):
    print("Installing K3s...")
    s.execute("curl -sfL https://get.k3s.io | sudo sh -")
    time.sleep(30)  # wait for K3s to start
    s.execute("sudo systemctl status k3s --no-pager | head -5")
    print("K3s installed.")
else:
    print("kubectl already available — K3s likely installed.")

# Make kubectl accessible without sudo
s.execute("sudo chmod 644 /etc/rancher/k3s/k3s.yaml")
s.execute("echo 'export KUBECONFIG=/etc/rancher/k3s/k3s.yaml' >> ~/.bashrc")

# Verify cluster
result = s.execute("sudo kubectl get nodes")
print(result)
# Should show: chameleon-node   Ready

NOT_INSTALLED
Command exited with status 0.
=== stdout ===
NOT_INSTALLED

(no stderr)
Installing K3s...
[INFO]  Finding release for channel stable
[INFO]  Using v1.34.6+k3s1 as release
[INFO]  Downloading hash https://github.com/k3s-io/k3s/releases/download/v1.34.6%2Bk3s1/sha256sum-amd64.txt
[INFO]  Downloading binary https://github.com/k3s-io/k3s/releases/download/v1.34.6%2Bk3s1/k3s
[INFO]  Verifying binary download
[INFO]  Installing k3s to /usr/local/bin/k3s
[INFO]  Skipping installation of SELinux RPM
[INFO]  Creating /usr/local/bin/kubectl symlink to k3s
[INFO]  Creating /usr/local/bin/crictl symlink to k3s
[INFO]  Skipping /usr/local/bin/ctr symlink to k3s, command exists in PATH at /usr/bin/ctr
[INFO]  Creating killall script /usr/local/bin/k3s-killall.sh
[INFO]  Creating uninstall script /usr/local/bin/k3s-uninstall.sh
[INFO]  env: Creating environment file /etc/systemd/system/k3s.service.env
[INFO]  systemd: Creating service file /etc/systemd/system/k3s.service
[INFO]  systemd

Created symlink /etc/systemd/system/multi-user.target.wants/k3s.service → /etc/systemd/system/k3s.service.


[INFO]  systemd: Starting k3s
● k3s.service - Lightweight Kubernetes
     Loaded: loaded (/etc/systemd/system/k3s.service; enabled; preset: enabled)
     Active: active (running) since Mon 2026-04-20 07:16:19 UTC; 34s ago
       Docs: https://k3s.io
    Process: 3367 ExecStartPre=/sbin/modprobe br_netfilter (code=exited, status=0/SUCCESS)
K3s installed.
NAME                      STATUS   ROLES           AGE   VERSION
node-integration-proj18   Ready    control-plane   45s   v1.34.6+k3s1
Command exited with status 0.
=== stdout ===
NAME                      STATUS   ROLES           AGE   VERSION
node-integration-proj18   Ready    control-plane   45s   v1.34.6+k3s1

(no stderr)


---
## Cell 8 — Clone all four team repos onto the VM

In [8]:
# Create a workspace directory
s.execute("mkdir -p /home/cc/proj18")

REPOS = [
    # (dir_name, github_url, branch)
    ("mealie",         "https://github.com/Sharvin27/mealie.git",                     "feature/ml-recommendations"),
    ("mealie-serving", "https://github.com/Sharvin27/mealie-serving.git",              "main"),
    ("mealie_als_training","https://github.com/Shashwatshah02/mealie_als_training.git","main"),
    ("mealie-data",    "https://github.com/brycemiranda/mealie-data-proj18.git",        "main"),
    ("mlops-devops",   "https://github.com/mahimamariah/proj18-mlops-devops.git",      "main"),
]

for dirname, url, branch in REPOS:
    dest = f"/home/cc/proj18/{dirname}"
    # Clone or pull if already exists
    result = s.execute(
        f"if [ -d {dest} ]; then "
        f"  git -C {dest} pull origin {branch} 2>&1 | tail -2; "
        f"else "
        f"  git clone -b {branch} {url} {dest} 2>&1 | tail -3; "
        f"fi"
    )
    print(f"{dirname}: {result}")

print("\nAll repos ready at /home/cc/proj18/")
result = s.execute("ls /home/cc/proj18/")
print(result)

Cloning into '/home/cc/proj18/mealie'...
mealie: Command exited with status 0.
=== stdout ===
Cloning into '/home/cc/proj18/mealie'...

(no stderr)
Cloning into '/home/cc/proj18/mealie-serving'...
mealie-serving: Command exited with status 0.
=== stdout ===
Cloning into '/home/cc/proj18/mealie-serving'...

(no stderr)
Cloning into '/home/cc/proj18/mealie-training'...
mealie-training: Command exited with status 0.
=== stdout ===
Cloning into '/home/cc/proj18/mealie-training'...

(no stderr)
Cloning into '/home/cc/proj18/mealie-data'...
mealie-data: Command exited with status 0.
=== stdout ===
Cloning into '/home/cc/proj18/mealie-data'...

(no stderr)
Cloning into '/home/cc/proj18/mlops-devops'...
mlops-devops: Command exited with status 0.
=== stdout ===
Cloning into '/home/cc/proj18/mlops-devops'...

(no stderr)

All repos ready at /home/cc/proj18/
mealie
mealie-data
mealie-serving
mealie-training
mlops-devops
Command exited with status 0.
=== stdout ===
mealie
mealie-data
mealie-servi

## Cell 9 — Create Kubernetes namespaces

In [9]:
namespaces = ["mealie-prod", "platform", "monitoring"]
for ns in namespaces:
    result = s.execute(
        f"sudo kubectl create namespace {ns} --dry-run=client -o yaml | sudo kubectl apply -f -"
    )
    print(f"{ns}: {result}")

# Verify
result = s.execute("sudo kubectl get namespaces")
print(result)

namespace/mealie-prod created
mealie-prod: Command exited with status 0.
=== stdout ===
namespace/mealie-prod created

(no stderr)
namespace/platform created
platform: Command exited with status 0.
=== stdout ===
namespace/platform created

(no stderr)
namespace/monitoring created
monitoring: Command exited with status 0.
=== stdout ===
namespace/monitoring created

(no stderr)
NAME              STATUS   AGE
default           Active   15m
kube-node-lease   Active   15m
kube-public       Active   15m
kube-system       Active   15m
mealie-prod       Active   12s
monitoring        Active   5s
platform          Active   8s
Command exited with status 0.
=== stdout ===
NAME              STATUS   AGE
default           Active   15m
kube-node-lease   Active   15m
kube-public       Active   15m
kube-system       Active   15m
mealie-prod       Active   12s
monitoring        Active   5s
platform          Active   8s

(no stderr)


## Cell 10 — Create Kubernetes Secrets

Secrets are NOT stored in any Git repo. We create them directly on the cluster.

In [10]:
# Database secret
s.execute(
    "sudo kubectl create secret generic db-secret "
    "--from-literal=POSTGRES_USER=mealie "
    "--from-literal=POSTGRES_PASSWORD=mealie_pass "
    "-n mealie-prod --dry-run=client -o yaml | sudo kubectl apply -f -"
)

# MinIO secret
s.execute(
    "sudo kubectl create secret generic minio-secret "
    "--from-literal=MINIO_ACCESS_KEY=minioadmin "
    "--from-literal=MINIO_SECRET_KEY=minioadmin123 "
    "-n platform --dry-run=client -o yaml | sudo kubectl apply -f -"
)

# MLflow secret
s.execute(
    "sudo kubectl create secret generic mlflow-db-secret "
    "--from-literal=POSTGRES_USER=mlflow "
    "--from-literal=POSTGRES_PASSWORD=mlflow123 "
    "-n platform --dry-run=client -o yaml | sudo kubectl apply -f -"
)

print("Secrets created (not stored in Git)")

secret/db-secret created
secret/minio-secret created
secret/mlflow-db-secret created
Secrets created (not stored in Git)


## Cell 11 — Create Shared ConfigMap

In [11]:
configmap = f"""apiVersion: v1
kind: ConfigMap
metadata:
  name: shared-env
  namespace: mealie-prod
data:
  INFERENCE_API_URL: 'http://inference-api.mealie-prod.svc.cluster.local:8000'
  MLFLOW_TRACKING_URI: 'http://mlflow.platform.svc.cluster.local:5000'
  MINIO_ENDPOINT: 'http://minio.platform.svc.cluster.local:9000'
  MINIO_BUCKET: 'mlflow-artifacts'
  MINIO_ACCESS_KEY: 'minioadmin'
  MINIO_SECRET_KEY: 'minioadmin123'
  POSTGRES_HOST: 'postgres.mealie-prod.svc.cluster.local'
  TAG_VECTOR_KEY: 'production/tag_to_vector.pkl'
"""

s.execute(f"cat > /tmp/shared-configmap.yaml << 'YAML'\n{configmap}\nYAML")
result = s.execute("sudo kubectl apply -f /tmp/shared-configmap.yaml")
print(result)

configmap/shared-env created
Command exited with status 0.
=== stdout ===
configmap/shared-env created

(no stderr)


---
## Cell 12 — Apply Mahima's K8s Manifests (Platform Services)

Deploy PostgreSQL, MinIO, MLflow first (platform layer).

In [12]:
# Fix secrets so Mahima's platform manifests can start

s.execute(
    "sudo kubectl create secret generic postgres-secret "
    "--from-literal=username=mealie "
    "--from-literal=password=mealie_pass "
    "-n platform --dry-run=client -o yaml | sudo kubectl apply -f -"
)

s.execute(
    "sudo kubectl create secret generic minio-secret "
    "--from-literal=username=minioadmin "
    "--from-literal=password=minioadmin123 "
    "-n platform --dry-run=client -o yaml | sudo kubectl apply -f -"
)

print(s.execute('sudo kubectl get secrets -n platform'))


secret/postgres-secret created
secret/minio-secret configured
NAME               TYPE     DATA   AGE
minio-secret       Opaque   2      16m
mlflow-db-secret   Opaque   2      16m
postgres-secret    Opaque   2      8s
Command exited with status 0.
=== stdout ===
NAME               TYPE     DATA   AGE
minio-secret       Opaque   2      16m
mlflow-db-secret   Opaque   2      16m
postgres-secret    Opaque   2      8s

(no stderr)


In [13]:
DEVOPS_DIR = "/home/cc/proj18/mlops-devops/infrastructure/k8s"

# Platform services: postgres, minio, mlflow
platform_manifests = [
    "postgres-statefulset.yaml",
    "minio-deployment.yaml",
    "mlflow-deployment.yaml",
]

for manifest in platform_manifests:
    path = f"{DEVOPS_DIR}/{manifest}"
    result = s.execute(
        f"if [ -f {path} ]; then "
        f"  sudo kubectl apply -f {path}; "
        f"else "
        f"  echo 'NOT FOUND: {manifest}'; "
        f"fi"
    )
    print(f"{manifest}: {result}")

print("\nWaiting 30s for platform services to start...")
time.sleep(30)
result = s.execute("sudo kubectl get pods -n platform")
print(result)

statefulset.apps/postgres created
service/postgres created
postgres-statefulset.yaml: Command exited with status 0.
=== stdout ===
statefulset.apps/postgres created
service/postgres created

(no stderr)
persistentvolumeclaim/minio-pvc created
deployment.apps/minio created
service/minio-service created
minio-deployment.yaml: Command exited with status 0.
=== stdout ===
persistentvolumeclaim/minio-pvc created
deployment.apps/minio created
service/minio-service created

(no stderr)
persistentvolumeclaim/mlflow-pvc created
deployment.apps/mlflow created
service/mlflow-service created
mlflow-deployment.yaml: Command exited with status 0.
=== stdout ===
persistentvolumeclaim/mlflow-pvc created
deployment.apps/mlflow created
service/mlflow-service created

(no stderr)

Waiting 30s for platform services to start...
NAME                      READY   STATUS    RESTARTS   AGE
minio-5b5d8dbfbc-jmvmg    1/1     Running   0          38s
mlflow-77964498b4-6ncvc   1/1     Running   0          34s
post

---
## Cell 12.5 — Download Food.com data, run ETL, train ALS model

This cell:
1. Downloads Food.com CSVs from Kaggle onto JupyterHub
2. Runs Bryce’s ETL (cleans + combines real + synthetic interactions)
3. Uploads train/val parquet to MinIO at `datasets/current/`
4. Clones Shashwat’s training repo on VM, patches train.py, builds image, trains ALS
5. Saves `production/tag_to_vector.pkl` to MinIO

> **Note:** Fill in your Kaggle token below before running.

In [ ]:
import ast, io, json as _json, os, subprocess, sys, time
from datetime import datetime
from pathlib import Path

# ── Kaggle credentials ──────────────────────────────────────────────
KAGGLE_TOKEN  = "KGAT_f62cafa6e07b5cccfd56c04c9dd73752"
KAGGLE_DATASET = "shuyangli94/food-com-recipes-and-user-interactions"

# ── Config ──────────────────────────────────────────────────────────
DATASET_VERSION = "current"
TRAINING_BUCKET = "training-data"
MODEL_BUCKET    = "mlflow-artifacts"
TAG_VECTOR_KEY  = "production/tag_to_vector.pkl"
TRAINING_REPO   = "https://github.com/Shashwatshah02/mealie_als_training.git"
TRAINING_DIR    = "/home/cc/proj18/mealie_als_training"
WEIGHT_MAP      = {5: 1.0, 4: 0.7, 3: 0.0, 2: -0.5, 1: -1.0}

# ── Step 1: Install deps on JupyterHub ──────────────────────────────
for pkg in ["pandas", "pyarrow", "numpy", "boto3", "kaggle"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import boto3, numpy as np, pandas as pd

# ── Step 2: Download Food.com CSVs from Kaggle ───────────────────────
os.environ["KAGGLE_TOKEN"] = KAGGLE_TOKEN
os.makedirs("/tmp/foodcom", exist_ok=True)
print("Downloading Food.com dataset from Kaggle...")
subprocess.check_call([
    sys.executable, "-m", "kaggle", "datasets", "download",
    "-d", KAGGLE_DATASET, "-p", "/tmp/foodcom", "--unzip"
])

RAW_RECIPES_CSV      = Path("/tmp/foodcom/RAW_recipes.csv")
RAW_INTERACTIONS_CSV = Path("/tmp/foodcom/RAW_interactions.csv")
if not RAW_RECIPES_CSV.exists():
    found = list(Path("/tmp/foodcom").rglob("RAW_recipes.csv"))
    if found:
        RAW_RECIPES_CSV      = found[0]
        RAW_INTERACTIONS_CSV = found[0].parent / "RAW_interactions.csv"
    else:
        raise FileNotFoundError("RAW_recipes.csv not found after Kaggle download")
print(f"Recipes:      {RAW_RECIPES_CSV}")
print(f"Interactions: {RAW_INTERACTIONS_CSV}")

# ── Step 3: Bryce ETL ────────────────────────────────────────────────
def parse_tags(value):
    if pd.isna(value): return []
    if isinstance(value, list): return [str(x) for x in value]
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except Exception:
            pass
    return []

print("Cleaning recipes...")
recipes = pd.read_csv(RAW_RECIPES_CSV, usecols=["id","name","tags","minutes","nutrition"])
recipes = recipes.dropna(subset=["id","name","tags","minutes"])
recipes["id"]      = recipes["id"].astype(str)
recipes["tags"]    = recipes["tags"].apply(parse_tags)
recipes["minutes"] = recipes["minutes"].clip(upper=480)
print(f"  {len(recipes):,} recipes")

print("Cleaning interactions...")
interactions = pd.read_csv(RAW_INTERACTIONS_CSV, usecols=["user_id","recipe_id","date","rating"])
interactions = interactions.dropna()
interactions["user_id"]   = interactions["user_id"].astype(str)
interactions["recipe_id"] = interactions["recipe_id"].astype(str)
interactions["weight"]    = interactions["rating"].map(WEIGHT_MAP)
interactions = interactions[interactions["weight"] != 0.0].copy()
interactions = interactions.sort_values("date")
print(f"  {len(interactions):,} real interactions")

print("Generating synthetic interactions...")
np.random.seed(42)
all_tags    = list({t for tags in recipes["tags"] for t in tags})
recipe_rows = recipes[["id","tags"]].values.tolist()
user_prefs  = {
    f"synth_{u}": set(np.random.choice(all_tags, size=np.random.randint(3,8), replace=False))
    for u in range(200)
}
rows = []
for _ in range(50_000):
    uid       = f"synth_{np.random.randint(0,200)}"
    rid, tags = recipe_rows[np.random.randint(0, len(recipe_rows))]
    overlap   = len(set(tags) & user_prefs[uid])
    if overlap >= 2:
        rating = int(np.random.choice([4,5], p=[0.4,0.6]))
    elif overlap == 1:
        rating = int(np.random.choice([3,4,5], p=[0.4,0.4,0.2]))
    else:
        rating = int(np.random.choice([1,2,3], p=[0.3,0.4,0.3]))
    w = WEIGHT_MAP.get(rating, 0.0)
    if w != 0.0:
        rows.append({"user_id": uid, "recipe_id": str(rid), "rating": rating, "weight": w, "date": "2025-01-01"})
synthetic = pd.DataFrame(rows)
print(f"  {len(synthetic):,} synthetic interactions")

combined = pd.concat([interactions, synthetic], ignore_index=True)
combined = combined.sort_values("date")
combined = combined.merge(recipes[["id","tags"]], left_on="recipe_id", right_on="id", how="left").drop(columns=["id"])
combined["tags"] = combined["tags"].apply(lambda x: x if isinstance(x, list) else [])
split = int(len(combined) * 0.8)
train = combined.iloc[:split].copy()
val   = combined.iloc[split:].copy()
print(f"  train: {len(train):,}  val: {len(val):,}")

# ── Step 4: Upload to MinIO ──────────────────────────────────────────
minio_client = boto3.client(
    "s3",
    endpoint_url=f"http://{floating_ip}:30900",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
)
try:
    minio_client.create_bucket(Bucket=TRAINING_BUCKET)
except Exception:
    pass

def upload_df(df, key):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    minio_client.put_object(Bucket=TRAINING_BUCKET, Key=key, Body=buf.getvalue())
    print(f"  uploaded s3://{TRAINING_BUCKET}/{key} ({len(df):,} rows)")

print("Uploading to MinIO...")
upload_df(recipes,      "processed/recipes_clean.parquet")
upload_df(interactions, "processed/interactions_clean.parquet")
upload_df(train,        f"datasets/{DATASET_VERSION}/train.parquet")
upload_df(val,          f"datasets/{DATASET_VERSION}/val.parquet")
meta = {"version": DATASET_VERSION, "train_rows": int(len(train)), "val_rows": int(len(val))}
minio_client.put_object(Bucket=TRAINING_BUCKET, Key=f"datasets/{DATASET_VERSION}/meta.json",
    Body=_json.dumps(meta, indent=2).encode())
print("MinIO upload complete:", meta)

# ── Step 5: Clone + patch Shashwat training repo on VM ───────────────
print("Cloning/updating training repo on VM...")
print(s.execute(
    f"if [ -d {TRAINING_DIR} ]; then git -C {TRAINING_DIR} pull --ff-only || true; "
    f"else git clone {TRAINING_REPO} {TRAINING_DIR}; fi"
))

patch_lines = [
    "from pathlib import Path",
    "import os",
    "path = Path('/home/cc/proj18/mealie_als_training/scripts/train.py')",
    "text = path.read_text()",
    "replacements = [",
    "    (\"        s3.download_file('training-data', 'datasets/v2_2026-04-04/train.parquet', f.name)\\n\",",
    "     \"        train_bucket = os.environ.get('TRAINING_BUCKET', 'training-data')\\n\"",
    "     \"        dataset_version = os.environ.get('DATASET_VERSION', 'current')\\n\"",
    "     \"        s3.download_file(train_bucket, f'datasets/{dataset_version}/train.parquet', f.name)\\n\"),",
    "    (\"        s3.download_file('training-data', 'datasets/v2_2026-04-04/val.parquet', f.name)\\n\",",
    "     \"        train_bucket = os.environ.get('TRAINING_BUCKET', 'training-data')\\n\"",
    "     \"        dataset_version = os.environ.get('DATASET_VERSION', 'current')\\n\"",
    "     \"        s3.download_file(train_bucket, f'datasets/{dataset_version}/val.parquet', f.name)\\n\"),",
    "    (\"        s3.upload_file(f.name, bucket, key)\\n\",",
    "     \"        try:\\n            s3.create_bucket(Bucket=bucket)\\n        except Exception:\\n            pass\\n        s3.upload_file(f.name, bucket, key)\\n\"),",
    "    (\"            save_to_minio(tag_to_vector, 'mlflow', 'production/tag_to_vector.pkl')\",",
    "     \"            save_to_minio(tag_to_vector, os.environ.get('MODEL_BUCKET', 'mlflow-artifacts'), os.environ.get('TAG_VECTOR_KEY', 'production/tag_to_vector.pkl'))\"),",
    "]",
    "for old, new in replacements:",
    "    if old in text:",
    "        text = text.replace(old, new)",
    "        print('Patched:', repr(old[:50]))",
    "    else:",
    "        print('WARN not found:', repr(old[:50]))",
    "path.write_text(text)",
    "print('Patch complete.')",
]
patch_script = "\n".join(patch_lines)
s.execute(f"printf '%s' {repr(patch_script)} > /tmp/patch_train.py")
print(s.execute("python3 /tmp/patch_train.py"))

# ── Step 6: Build image + import into K3s ────────────────────────────
print("Building training Docker image on VM (takes ~3 min)...")
print(s.execute(
    f"bash -lc 'cd {TRAINING_DIR} && sudo docker build -t mealie-als-training:integration-rerun . 2>&1 | tail -5'"
))
print("Importing into K3s containerd...")
print(s.execute(
    "sudo docker save mealie-als-training:integration-rerun | sudo k3s ctr images import -"
))

# ── Step 7: Run ALS training ─────────────────────────────────────────
print("Running ALS training (may take 5-15 min on full dataset)...")
train_cmd = (
    "sudo docker run --rm --network host "
    "-e MLFLOW_TRACKING_URI=http://127.0.0.1:30500 "
    "-e MLFLOW_S3_ENDPOINT_URL=http://127.0.0.1:30900 "
    "-e AWS_ACCESS_KEY_ID=minioadmin "
    "-e AWS_SECRET_ACCESS_KEY=minioadmin123 "
    "-e MINIO_ENDPOINT=http://127.0.0.1:30900 "
    "-e MINIO_ACCESS_KEY=minioadmin "
    "-e MINIO_SECRET_KEY=minioadmin123 "
    f"-e TRAINING_BUCKET={TRAINING_BUCKET} "
    f"-e DATASET_VERSION={DATASET_VERSION} "
    f"-e MODEL_BUCKET={MODEL_BUCKET} "
    f"-e TAG_VECTOR_KEY={TAG_VECTOR_KEY} "
    "mealie-als-training:integration-rerun"
)
print(s.execute(train_cmd))
print("Bootstrap complete — production/tag_to_vector.pkl saved to MinIO.")

## Cell 13 — Build and deploy the Inference API

In [14]:
SERVING_DIR = "/home/cc/proj18/mealie-serving/serving"

# Build the inference API image on the VM
print("Building inference API Docker image...")
result = s.execute(
    f"cd {SERVING_DIR} && "
    f"sudo docker build -f Dockerfile.cached -t mealie-inference:latest . 2>&1 | tail -5"
)
print(result)

# Import the image into K3s containerd
print("Importing image into K3s...")
result = s.execute(
    "sudo docker save mealie-inference:latest | sudo k3s ctr images import -"
)
print(result)

Building inference API Docker image...
#12 exporting manifest list sha256:b13c867361380d6e27553f1ac8b26c27db8c53a835c49a60a86e3083e14ed23c 0.0s done
#12 naming to docker.io/library/mealie-inference:latest done
#12 unpacking to docker.io/library/mealie-inference:latest
#12 unpacking to docker.io/library/mealie-inference:latest 4.2s done
#12 DONE 17.8s
Command exited with status 0.
=== stdout ===
#12 exporting manifest list sha256:b13c867361380d6e27553f1ac8b26c27db8c53a835c49a60a86e3083e14ed23c 0.0s done
#12 naming to docker.io/library/mealie-inference:latest done
#12 unpacking to docker.io/library/mealie-inference:latest
#12 unpacking to docker.io/library/mealie-inference:latest 4.2s done
#12 DONE 17.8s

(no stderr)
Importing image into K3s...
docker.io/library/mealie inference:lates	saved	
application/vnd.oci.image.index.v1+json sha256:b13c867361380d6e27553f1ac8b26c27db8c53a835c49a60a86e3083e14ed23c
Importing	elapsed: 10.5s	total:   0.0 B	(0.0 B/s)	
Command exited with status 0.
=== st

In [15]:
print(s.execute("sudo docker images | grep mealie-inference"))


mealie-inference:latest   b13c86736138        486MB          115MB        
Command exited with status 0.
=== stdout ===
mealie-inference:latest   b13c86736138        486MB          115MB

=== stderr ===



## Cell 14 — Create the Inference API Kubernetes Deployment

In [16]:
configmap = """apiVersion: v1
kind: ConfigMap
metadata:
  name: shared-env
  namespace: mealie-prod
data:
  INFERENCE_API_URL: 'http://inference-api.mealie-prod.svc.cluster.local:8000'
  MLFLOW_TRACKING_URI: 'http://mlflow-service.platform.svc.cluster.local:5000'
  MINIO_ENDPOINT: 'http://minio-service.platform.svc.cluster.local:9000'
  MINIO_BUCKET: 'mlflow-artifacts'
  MINIO_ACCESS_KEY: 'minioadmin'
  MINIO_SECRET_KEY: 'minioadmin123'
  POSTGRES_HOST: 'postgres.platform.svc.cluster.local'
  TAG_VECTOR_KEY: 'production/tag_to_vector.pkl'
"""

s.execute(f"cat > /tmp/shared-configmap.yaml << 'YAML'\n{configmap}\nYAML")
print(s.execute("sudo kubectl apply -f /tmp/shared-configmap.yaml"))


configmap/shared-env configured
Command exited with status 0.
=== stdout ===
configmap/shared-env configured

(no stderr)


In [17]:
inference_manifest = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: inference-api
  namespace: mealie-prod
spec:
  replicas: 1
  selector:
    matchLabels:
      app: inference-api
  template:
    metadata:
      labels:
        app: inference-api
    spec:
      containers:
      - name: inference-api
        image: mealie-inference:latest
        imagePullPolicy: Never
        ports:
        - containerPort: 8000
        envFrom:
        - configMapRef:
            name: shared-env
        resources:
          requests:
            cpu: 250m
            memory: 256Mi
          limits:
            cpu: 500m
            memory: 512Mi
---
apiVersion: v1
kind: Service
metadata:
  name: inference-api
  namespace: mealie-prod
spec:
  selector:
    app: inference-api
  ports:
  - port: 8000
    targetPort: 8000
    nodePort: 30800
  type: NodePort
"""

s.execute(f"cat > /tmp/inference-deployment.yaml << 'YAML'\n{inference_manifest}\nYAML")
result = s.execute("sudo kubectl apply -f /tmp/inference-deployment.yaml")
print(result)

deployment.apps/inference-api created
service/inference-api created
Command exited with status 0.
=== stdout ===
deployment.apps/inference-api created
service/inference-api created

(no stderr)


In [18]:
print(s.execute("sudo kubectl get pods -n mealie-prod"))
print(s.execute("sudo kubectl get svc -n mealie-prod"))


NAME                             READY   STATUS    RESTARTS   AGE
inference-api-6bcf7864d4-4q9wz   1/1     Running   0          4s
Command exited with status 0.
=== stdout ===
NAME                             READY   STATUS    RESTARTS   AGE
inference-api-6bcf7864d4-4q9wz   1/1     Running   0          4s

(no stderr)
NAME            TYPE       CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE
inference-api   NodePort   10.43.205.143   <none>        8000:30800/TCP   8s
Command exited with status 0.
=== stdout ===
NAME            TYPE       CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE
inference-api   NodePort   10.43.205.143   <none>        8000:30800/TCP   8s

(no stderr)


## Cell 15 — Build and deploy Mealie (your fork)

In [9]:
s.refresh()
s.show(type="widget")
s.check_connectivity()


Attribute,node-integration-proj18
Id,dafd56b6-387f-45ca-bc07-8a58a50774ac
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:833254fc-59df-4262-a484-2393f8016a41
Addresses,sharednet1: IP: 10.56.1.141 (v4) Type: fixed MAC: fa:16:3e:8b:20:fa IP: 129.114.26.214 (v4) Type: floating MAC: fa:16:3e:8b:20:fa
Network Name,sharednet1
Created At,2026-04-20T07:12:06Z
Keypair,sg9469_nyu_edu-jupyter
Reservation Id,None
Host Id,951daf2a5ba2285d35cce6efd97dfeafbaf2e84b1605a1aaba7bfdd3


Checking connectivity to 129.114.26.214 port 22.


Connection successful


In [28]:
print(s.execute("hostname && whoami"))


node-integration-proj18
cc
Command exited with status 0.
=== stdout ===
node-integration-proj18
cc

(no stderr)


In [29]:
print(s.execute("sudo docker images | grep mealie-custom || true"))


Command exited with status 0.
(no stdout)
=== stderr ===



In [11]:
MEALIE_DIR = "/home/cc/proj18/mealie"

print("Building Mealie custom image...")
result = s.execute(
    f"bash -lc 'set -o pipefail; cd {MEALIE_DIR} && sudo docker build --file docker/Dockerfile -t mealie-custom:latest .'"
)
print(result)

print("Importing into K3s...")
result = s.execute(
    "sudo docker save mealie-custom:latest | sudo k3s ctr images import -"
)
print(result)
print("Mealie image ready")


Building Mealie custom image...


/opt/conda/lib/python3.12/site-packages/paramiko/client.py:885: UserWarning: Unknown ssh-ed25519 host key for 129.114.26.214: b'4da268233d1eb184b2d0bf6e2e24566f'
  warnings.warn(
#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 30B 0.1s
#1 transferring dockerfile: 4.87kB 0.1s done
#1 DONE 0.2s

#2 [internal] load metadata for docker.io/library/node:24@sha256:33cf7f057918860b043c307751ef621d74ac96f875b79b6724dcebf2dfd0db6d
#2 DONE 1.0s

#3 [internal] load metadata for docker.io/library/python:3.12-slim@sha256:7026274c107626d7e940e0e5d6730481a4600ae95d5ca7eb532dd4180313fea9
#3 DONE 1.0s

#4 [internal] load .dockerignore
#4 transferring context: 33B 0.0s
#4 transferring context: 359B 0.1s done
#4 DONE 0.1s

#5 [internal] load build context
#5 DONE 0.0s

#6 [python-base 1/2] FROM docker.io/library/python:3.12-slim@sha256:7026274c107626d7e940e0e5d6730481a4600ae95d5ca7eb532dd4180313fea9
#6 resolve docker.

KeyboardInterrupt: 

In [32]:
print(s.execute("sudo docker images | grep mealie-custom || true"))
print(s.execute("sudo kubectl get ns"))


mealie-custom:latest      1d53175525e9       1.72GB          411MB        
Command exited with status 0.
=== stdout ===
mealie-custom:latest      1d53175525e9       1.72GB          411MB

=== stderr ===

NAME              STATUS   AGE
default           Active   108m
kube-node-lease   Active   108m
kube-public       Active   108m
kube-system       Active   108m
mealie-prod       Active   92m
monitoring        Active   92m
platform          Active   92m
Command exited with status 0.
=== stdout ===
NAME              STATUS   AGE
default           Active   108m
kube-node-lease   Active   108m
kube-public       Active   108m
kube-system       Active   108m
mealie-prod       Active   92m
monitoring        Active   92m
platform          Active   92m

(no stderr)


## Cell 16 — Apply Mealie K8s Deployment



In [33]:
mealie_deploy = f"""apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: mealie-data-pvc
  namespace: mealie-prod
spec:
  accessModes: ["ReadWriteOnce"]
  resources:
    requests:
      storage: 5Gi
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mealie
  namespace: mealie-prod
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mealie
  template:
    metadata:
      labels:
        app: mealie
    spec:
      containers:
      - name: mealie
        image: mealie-custom:latest
        imagePullPolicy: Never
        ports:
        - containerPort: 9000
        env:
        - name: ALLOW_SIGNUP
          value: "true"
        - name: BASE_URL
          value: "http://{floating_ip}:30090"
        - name: DB_ENGINE
          value: postgres
        - name: POSTGRES_SERVER
          value: postgres.platform.svc.cluster.local
        - name: POSTGRES_PORT
          value: "5432"
        - name: POSTGRES_DB
          value: mealie
        - name: POSTGRES_USER
          valueFrom:
            secretKeyRef:
              name: postgres-secret
              key: username
              optional: false
        - name: POSTGRES_PASSWORD
          valueFrom:
            secretKeyRef:
              name: postgres-secret
              key: password
              optional: false
        - name: INFERENCE_API_URL
          valueFrom:
            configMapKeyRef:
              name: shared-env
              key: INFERENCE_API_URL
        volumeMounts:
        - mountPath: /app/data
          name: mealie-data
        resources:
          requests:
            cpu: 500m
            memory: 512Mi
          limits:
            cpu: 1000m
            memory: 1Gi
      volumes:
      - name: mealie-data
        persistentVolumeClaim:
          claimName: mealie-data-pvc
---
apiVersion: v1
kind: Service
metadata:
  name: mealie-service
  namespace: mealie-prod
spec:
  type: NodePort
  selector:
    app: mealie
  ports:
  - port: 9000
    targetPort: 9000
    nodePort: 30090
"""

s.execute(f"cat > /tmp/mealie-deployment.yaml << 'YAML'\n{mealie_deploy}\nYAML")
print(s.execute("sudo kubectl apply -f /tmp/mealie-deployment.yaml"))
print(s.execute("sudo kubectl get pods -n mealie-prod"))
print(s.execute("sudo kubectl get svc -n mealie-prod"))


persistentvolumeclaim/mealie-data-pvc created
deployment.apps/mealie created
service/mealie-service created
Command exited with status 0.
=== stdout ===
persistentvolumeclaim/mealie-data-pvc created
deployment.apps/mealie created
service/mealie-service created

(no stderr)
NAME                             READY   STATUS             RESTARTS         AGE
inference-api-6bcf7864d4-4q9wz   0/1     CrashLoopBackOff   16 (3m59s ago)   61m
mealie-6c8fcdddd7-jk7xv          0/1     Pending            0                4s
Command exited with status 0.
=== stdout ===
NAME                             READY   STATUS             RESTARTS         AGE
inference-api-6bcf7864d4-4q9wz   0/1     CrashLoopBackOff   16 (3m59s ago)   61m
mealie-6c8fcdddd7-jk7xv          0/1     Pending            0                4s

(no stderr)
NAME             TYPE       CLUSTER-IP      EXTERNAL-IP   PORT(S)          AGE
inference-api    NodePort   10.43.205.143   <none>        8000:30800/TCP   61m
mealie-service   NodePort 

In [34]:
print(s.execute(
    "sudo kubectl create secret generic postgres-secret "
    "--from-literal=username=mealie "
    "--from-literal=password=mealie_pass "
    "-n mealie-prod --dry-run=client -o yaml | sudo kubectl apply -f -"
))


secret/postgres-secret created
Command exited with status 0.
=== stdout ===
secret/postgres-secret created

(no stderr)


In [35]:
print(s.execute("sudo kubectl get pvc -n mealie-prod"))
print(s.execute("sudo kubectl describe pod -n mealie-prod -l app=mealie"))


NAME              STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS   VOLUMEATTRIBUTESCLASS   AGE
mealie-data-pvc   Bound    pvc-fbdbd72b-908d-44af-ab70-ccced3c992ef   5Gi        RWO            local-path     <unset>                 2m1s
Command exited with status 0.
=== stdout ===
NAME              STATUS   VOLUME                                     CAPACITY   ACCESS MODES   STORAGECLASS   VOLUMEATTRIBUTESCLASS   AGE
mealie-data-pvc   Bound    pvc-fbdbd72b-908d-44af-ab70-ccced3c992ef   5Gi        RWO            local-path     <unset>                 2m1s

(no stderr)
Name:             mealie-6c8fcdddd7-jk7xv
Namespace:        mealie-prod
Priority:         0
Service Account:  default
Node:             node-integration-proj18/10.56.1.141
Start Time:       Mon, 20 Apr 2026 09:05:14 +0000
Labels:           app=mealie
                  pod-template-hash=6c8fcdddd7
Annotations:      <none>
Status:           Running
IP:               10.42.0.28
IPs:
 

## Cell 17 — Apply CronJobs (ETL + Retrain)

In [ ]:
# Cell 17 - Apply CronJobs from mealie repo (ETL + Retrain + Eval)
MEALIE_DIR = "/home/cc/proj18/mealie"

# Pull latest so we have the corrected manifests from dev/cronjobs/
s.execute(f"git -C {MEALIE_DIR} pull origin feature/ml-recommendations 2>&1 | tail -3")

cronjobs = [
    "mealie-prod-batch-compile-cronjob.yaml",
    "mealie-prod-monthly-retrain-cronjob.yaml",
    "mealie-prod-nightly-eval-cronjob.yaml",
]

for cj in cronjobs:
    path = f"{MEALIE_DIR}/dev/cronjobs/{cj}"
    result = s.execute(f"sudo kubectl apply -f {path}")
    print(f"{cj}: {result}")

print(s.execute("sudo kubectl get cronjobs -n mealie-prod"))


## Cell 17.5 - Deploy Monitoring Stack
Deploys Prometheus, Grafana, kube-state-metrics into `monitoring` namespace.
Patches CPU requests down to fit single-node Chameleon VM.
- Grafana: `http://<floating-ip>:30300` (admin / admin123)
- Prometheus: `http://<floating-ip>:30091`

In [ ]:
# Cell 17.5 - Deploy Monitoring Stack
import json as _json
MEALIE_DIR = "/home/cc/proj18/mealie"
MON_DIR = f"{MEALIE_DIR}/dev/monitoring"

print("--- RBAC ---")
for f in ["prometheus-rbac.yaml", "kube-state-metrics-rbac.yaml"]:
    print(s.execute(f"sudo kubectl apply -f {MON_DIR}/{f}"))

print("--- ConfigMaps ---")
for f in ["prometheus-configmap.yaml", "grafana-configmap.yaml", "alertmanager-configmap.yaml"]:
    print(s.execute(f"sudo kubectl apply -f {MON_DIR}/{f}"))

print("--- Deployments ---")
for f in ["kube-state-metrics.yaml", "prometheus-deployment.yaml", "grafana-deployment.yaml"]:
    print(s.execute(f"sudo kubectl apply -f {MON_DIR}/{f}"))

print("--- HPA ---")
print(s.execute(f"sudo kubectl apply -f {MON_DIR}/inference-api-hpa.yaml"))

# Reduce CPU requests to fit single-node Chameleon VM
patches = [
    ("mealie",        "mealie-prod", "100m", "512Mi"),
    ("inference-api", "mealie-prod", "50m",  "256Mi"),
    ("prometheus",    "monitoring",  "100m", "512Mi"),
    ("grafana",       "monitoring",  "10m",  "128Mi"),
]
for name, ns, cpu, mem in patches:
    patch = _json.dumps({"spec":{"template":{"spec":{"containers":[{"name":name,"resources":{"requests":{"cpu":cpu,"memory":mem}}}]}}}})
    s.execute(f"sudo kubectl patch deployment {name} -n {ns} --patch '{patch}'")
    print(f"Patched {name}: cpu={cpu} mem={mem}")

print("Waiting 60s for monitoring pods...")
time.sleep(60)
print(s.execute("sudo kubectl get pods -n monitoring"))


## Cell 17.6 - Staging + Canary Environments
Creates `mealie-staging` and `mealie-canary` namespaces each with their own inference-api.
- staging → `staging/tag_to_vector.pkl` in MinIO (NodePort 30801)
- canary  → `canary/tag_to_vector.pkl` in MinIO  (NodePort 30802)
- prod    → `production/tag_to_vector.pkl`        (NodePort 30800)

Monthly-retrain is patched to save new models to `staging/` instead of `production/` directly.

In [ ]:
# Cell 17.6 - Staging + Canary Environments

for ns in ["mealie-staging", "mealie-canary"]:
    s.execute(f"sudo kubectl create namespace {ns} --dry-run=client -o yaml | sudo kubectl apply -f -")
    s.execute(
        f"sudo kubectl get secret postgres-secret -n mealie-prod -o yaml "
        f"| sed 's/namespace: mealie-prod/namespace: {ns}/' "
        f"| sudo kubectl apply -f -"
    )
    print(f"Namespace {ns} ready")

env_base = {
    "MLFLOW_TRACKING_URI": "http://mlflow-service.platform.svc.cluster.local:5000",
    "MINIO_ENDPOINT":      "http://minio-service.platform.svc.cluster.local:9000",
    "MINIO_BUCKET":        "mlflow-artifacts",
    "MINIO_ACCESS_KEY":    "minioadmin",
    "MINIO_SECRET_KEY":    "minioadmin123",
    "POSTGRES_HOST":       "postgres.platform.svc.cluster.local",
}

envs = {
    "mealie-staging": {**env_base,
        "INFERENCE_API_URL": "http://inference-api.mealie-staging.svc.cluster.local:8000",
        "TAG_VECTOR_KEY":    "staging/tag_to_vector.pkl"},
    "mealie-canary": {**env_base,
        "INFERENCE_API_URL": "http://inference-api.mealie-canary.svc.cluster.local:8000",
        "TAG_VECTOR_KEY":    "canary/tag_to_vector.pkl"},
}

for ns, env in envs.items():
    args = " ".join([f"--from-literal={k}={v}" for k, v in env.items()])
    s.execute(f"sudo kubectl create configmap shared-env -n {ns} {args} --dry-run=client -o yaml | sudo kubectl apply -f -")

ports = {"mealie-staging": 30801, "mealie-canary": 30802}
for ns, nodeport in ports.items():
    manifest = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: inference-api
  namespace: {ns}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: inference-api
  template:
    metadata:
      labels:
        app: inference-api
    spec:
      containers:
      - name: inference-api
        image: mealie-inference:latest
        imagePullPolicy: Never
        ports:
        - containerPort: 8000
        envFrom:
        - configMapRef:
            name: shared-env
        resources:
          requests:
            cpu: 50m
            memory: 128Mi
          limits:
            cpu: 500m
            memory: 512Mi
---
apiVersion: v1
kind: Service
metadata:
  name: inference-api
  namespace: {ns}
spec:
  selector:
    app: inference-api
  ports:
  - port: 8000
    targetPort: 8000
    nodePort: {nodeport}
  type: NodePort"""
    s.execute(f"cat > /tmp/inf-{ns}.yaml << 'YAML'\n{manifest}\nYAML")
    result = s.execute(f"sudo kubectl apply -f /tmp/inf-{ns}.yaml")
    print(f"{ns}: {result}")

# Retrain now saves to staging/ first, not production/ directly
s.execute(
    "sudo kubectl patch cronjob monthly-retrain -n mealie-prod "
    "--type='json' "
    "-p='[{"op":"replace","path":"/spec/jobTemplate/spec/template/spec/containers/0/env/3/value","value":"staging/tag_to_vector.pkl"}]'"
)

print(s.execute("sudo kubectl get pods -n mealie-staging"))
print(s.execute("sudo kubectl get pods -n mealie-canary"))


## Cell 17.7 - Model Promoter CronJob (automated CI/CD)
Runs every 6 hours. Reads MLflow `nightly-eval` metrics and:
- Promotes `staging -> canary -> production` if eval passes
- Auto-rollbacks production from backup if inference goes down

Promotion rules:
- `train_rows > 0` (dataset has training data)
- `tag_vector_exists = 1` (model artifact present)
- `inference_status = 200` (inference API healthy)

In [ ]:
# Cell 17.7 - Model Promoter CronJob
promoter = """\
apiVersion: batch/v1
kind: CronJob
metadata:
  name: model-promoter
  namespace: mealie-prod
spec:
  schedule: "0 */6 * * *"
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 3
  jobTemplate:
    spec:
      backoffLimit: 1
      template:
        spec:
          restartPolicy: OnFailure
          containers:
            - name: model-promoter
              image: python:3.11-slim
              imagePullPolicy: IfNotPresent
              envFrom:
                - configMapRef:
                    name: shared-env
              command: ["/bin/sh", "-lc"]
              args:
                - |
                  pip install --no-cache-dir boto3 mlflow >/tmp/pip.log 2>&1
                  python - <<'PY'
                  import os,sys
                  import boto3,mlflow
                  endpoint=os.environ["MINIO_ENDPOINT"]
                  access=os.environ["MINIO_ACCESS_KEY"]
                  secret=os.environ["MINIO_SECRET_KEY"]
                  bucket="mlflow-artifacts"
                  s3=boto3.client("s3",endpoint_url=endpoint,aws_access_key_id=access,aws_secret_access_key=secret)
                  def cp(src,dst):
                      obj=s3.get_object(Bucket=bucket,Key=src)
                      s3.put_object(Bucket=bucket,Key=dst,Body=obj["Body"].read())
                      print(f"Copied {src} -> {dst}")
                  def ex(k):
                      try: s3.head_object(Bucket=bucket,Key=k); return True
                      except: return False
                  mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
                  client=mlflow.tracking.MlflowClient()
                  exp=client.get_experiment_by_name("nightly-eval")
                  if not exp: print("No eval experiment yet"); sys.exit(0)
                  runs=client.search_runs([exp.experiment_id],order_by=["start_time DESC"],max_results=1)
                  if not runs: print("No runs yet"); sys.exit(0)
                  m=runs[0].data.metrics
                  ok=int(m.get("inference_status",0))==200
                  tv=int(m.get("tag_vector_exists",0))==1
                  tr=int(m.get("train_rows",0))
                  print(f"Eval: inference_ok={ok} tag_vec={tv} train_rows={tr}")
                  if not ok:
                      if ex("production/tag_to_vector.pkl.bak"): cp("production/tag_to_vector.pkl.bak","production/tag_to_vector.pkl"); print("ROLLBACK done")
                      else: print("ROLLBACK needed but no backup found")
                      sys.exit(0)
                  if ex("staging/tag_to_vector.pkl") and tv and tr>0:
                      if ex("canary/tag_to_vector.pkl"):
                          if ex("production/tag_to_vector.pkl"): cp("production/tag_to_vector.pkl","production/tag_to_vector.pkl.bak")
                          cp("canary/tag_to_vector.pkl","production/tag_to_vector.pkl"); print("PROMOTED canary->production")
                      cp("staging/tag_to_vector.pkl","canary/tag_to_vector.pkl"); print("PROMOTED staging->canary")
                  else: print(f"Not ready: staging={ex(chr(39)+"staging/tag_to_vector.pkl"+chr(39))} rows={tr}")
                  PY
              resources:
                requests:
                  cpu: "50m"
                  memory: "128Mi"
                limits:
                  cpu: "200m"
                  memory: "256Mi"
"""
s.execute(f"cat > /tmp/model-promoter.yaml << 'YAML'\n{promoter}\nYAML")
print(s.execute("sudo kubectl apply -f /tmp/model-promoter.yaml"))
print(s.execute("sudo kubectl get cronjob model-promoter -n mealie-prod"))


## Cell 18 — Wait for pods and verify everything is running

In [ ]:
print("Waiting 60s for all pods to start...")
time.sleep(60)

print("=== All pods ===")
result = s.execute("sudo kubectl get pods --all-namespaces")
print(result)

print("\n=== Services ===")
result = s.execute("sudo kubectl get services --all-namespaces")
print(result)

## Cell 19 — Generate stub model artifacts (if MinIO does not have them yet)

In [ ]:
# Run Shashwat's stub model generator to populate MinIO
# This gives the inference API something to load until the real model is trained

stub_script = '''
import numpy as np, joblib, os, boto3, io

np.random.seed(42)
tags = [
    'italian','vegetarian','asian','quick','comfort-food','pasta',
    'mexican','desserts','15-minutes-or-less','30-minutes-or-less',
    '60-minutes-or-less','healthy','meat','low-calorie','chicken',
    'beef','seafood','soups-stews','salads','breakfast','side-dishes',
    'appetizers','sandwiches','pizza','indian','thai','chinese','greek',
    'french','spanish','4-hours-or-less','easy','one-pot-meals',
    'high-protein','low-fat','low-carb','gluten-free','dairy-free',
    'vegan','kid-friendly','spicy','sweet','savory','baking','grilling',
    'slow-cooker','stir-fry','roasting'
]

tag_to_vector = {t: np.random.randn(50).astype(np.float32) for t in tags}
joblib.dump(tag_to_vector, '/tmp/tag_to_vector.pkl')

# Upload to MinIO
try:
    s3 = boto3.client('s3',
        endpoint_url='http://minio.platform.svc.cluster.local:9000',
        aws_access_key_id='minioadmin',
        aws_secret_access_key='minioadmin123')
    # Create bucket if needed
    try: s3.create_bucket(Bucket='mlflow-artifacts')
    except: pass
    s3.upload_file('/tmp/tag_to_vector.pkl', 'mlflow-artifacts', 'production/tag_to_vector.pkl')
    print('Uploaded tag_to_vector.pkl to MinIO at production/tag_to_vector.pkl')
except Exception as e:
    print(f'MinIO upload failed: {e}')
    print('Saving locally to /home/cc/artifacts/ instead')
    os.makedirs('/home/cc/artifacts', exist_ok=True)
    joblib.dump(tag_to_vector, '/home/cc/artifacts/tag_to_vector.pkl')
    print('Saved locally — mount /home/cc/artifacts as volume in inference-api pod')

print('Done! Inference API can now start loading vectors.')
'''

s.execute("pip3 install numpy joblib boto3 2>&1 | tail -3")
s.execute(f"python3 -c \"{stub_script.replace(chr(34), chr(39))}\"")

# Simpler: just write the script to a file and run it
s.execute("cat > /tmp/make_stub.py << 'PYEOF'\n" + stub_script + "\nPYEOF")
result = s.execute("python3 /tmp/make_stub.py")
print(result)

## Cell 20 — Quick health checks on all endpoints

In [ ]:
print("=== Inference API health ===")
result = s.execute("curl -s http://localhost:30800/health 2>/dev/null || echo 'Not ready yet'")
print(result)

print("\n=== Test /tag-vector endpoint ===")
result = s.execute(
    "curl -s -X POST http://localhost:30800/tag-vector "
    "-H 'Content-Type: application/json' "
    "-d '{\"tags\":[\"italian\",\"vegetarian\"]}' 2>/dev/null || echo 'Not ready yet'"
)
print(result)

print("\n=== Mealie UI ===")
result = s.execute("curl -s -o /dev/null -w '%{http_code}' http://localhost:30090 2>/dev/null || echo 'Not ready yet'")
print(f"Mealie HTTP status: {result}")

print("\n=== MLflow UI ===")
result = s.execute("curl -s -o /dev/null -w '%{http_code}' http://localhost:30500 2>/dev/null || echo 'Not ready yet'")
print(f"MLflow HTTP status: {result}")

## Cell 21 — Print all access URLs and SSH command

In [ ]:
print("=" * 60)
print("  INTEGRATION SYSTEM — ACCESS URLS")
print("=" * 60)
print(f"  Mealie UI:        http://{floating_ip}:30090")
print(f"  Inference API:    http://{floating_ip}:30800")
print(f"  Inference health: http://{floating_ip}:30800/health")
print(f"  MLflow UI:        http://{floating_ip}:30500")
print(f"  MinIO Console:    http://{floating_ip}:30900")
print(f"  Prometheus:       http://{floating_ip}:30090/metrics (inference API)")
print("=" * 60)
print(f"\nSSH to VM (from your local terminal):")
print(f"  ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print(f"\nOnce SSH'd in:")
print(f"  sudo kubectl get pods --all-namespaces   # check all pods")
print(f"  sudo kubectl logs deployment/inference-api -n mealie-prod  # inference logs")
print(f"  sudo kubectl logs deployment/mealie -n mealie-prod          # mealie logs")
print("\nSubmission checklist:")
print("  [ ] Mealie UI loads in browser")
print("  [ ] New user sees /preferences onboarding page")
print("  [ ] After onboarding, Recommended for You panel visible")
print("  [ ] /tag-vector endpoint returns 50-dim vector")
print("  [ ] /recommend endpoint returns ranked list")
print("  [ ] MLflow UI accessible with training runs")
print("  [ ] CronJobs visible: kubectl get cronjobs --all-namespaces")

## Cell 22 — Open port 30090 and 30800 for external access

In [ ]:
# Open all firewall ports for external access
ports = [30090, 30800, 30500, 30900, 30300, 30091, 30903, 30801, 30802]
for port in ports:
    s.execute(f"sudo iptables -I INPUT -p tcp --dport {port} -j ACCEPT")
    print(f"Opened port {port}")

print(f"\nAll services externally accessible:")
print(f"  Mealie UI:     http://{floating_ip}:30090")
print(f"  Inference API: http://{floating_ip}:30800")
print(f"  MLflow:        http://{floating_ip}:30500")
print(f"  MinIO:         http://{floating_ip}:30900")
print(f"  Grafana:       http://{floating_ip}:30300  (admin/admin123)")
print(f"  Prometheus:    http://{floating_ip}:30091")
print(f"  Staging API:   http://{floating_ip}:30801")
print(f"  Canary API:    http://{floating_ip}:30802")
